# Task 03 — Data Preprocessing and Feature Engineering

### SmartCare Hospital AI Dataset | Option C – Disease Risk Classification
**CCS3440 Artificial Intelligence Coursework**  
* **ST NAME:** Kaveesha Dilshan (CIT-23-02-0127) | `@Kaveesha23dil`  
* **Role:** Data Preprocessing & Feature Engineering  
* **Target Variable:** `disease_risk_level` (Low / Medium / High)

---

### Pipeline Governance & Auditability
This notebook follows a production-grade, section-wise preprocessing pipeline. Every transformation is:
1. **Justified clinically & mathematically** prior to execution.
2. **Audited with shape/count logging** before and after each transformation.
3. **Designed to eliminate Data Leakage** by strictly segregating predictive features from administrative, financial, and future-leakage attributes.
---
**Pipeline Order:**  
`Setup` ➔ `Missing Values` ➔ `Duplicates` ➔ `Outlier Inspection` ➔ `Feature Engineering` ➔ `Feature Selection` ➔ `Categorical Encoding` ➔ `Train/Test Split & Scaling` ➔ `Artifact Export`

## Section 1 — Setup and Data Load
Import required dependencies and load the raw dataset. We retain an unmodified copy (`df_raw`) throughout the runtime for auditability during the viva examination.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path

# Pandas display settings for complete visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RANDOM_STATE = 42

# Load raw dataset
DATA_PATH = '/smartcare_ai_dataset_1000.csv'  # Adjust path if needed
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset shape: (1000, 33)
Columns: ['record_id', 'patient_id', 'age', 'gender', 'blood_group', 'department', 'diagnosis', 'appointment_date', 'waiting_days', 'previous_appointments', 'missed_previous_appointments', 'appointment_status', 'admitted', 'room_type', 'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count', 'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr', 'payment_status', 'payment_method', 'no_show', 'readmitted_30_days', 'disease_risk_level']


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


In [4]:
# Quick structural check before modifications
print("Data Types Overview:")
print(df.dtypes)

print("\nTarget Distribution (disease_risk_level):")
print(df['disease_risk_level'].value_counts())
print("\nPercentage Distribution (%):")
print((df['disease_risk_level'].value_counts(normalize=True).round(3) * 100).astype(str) + '%')

Data Types Overview:
record_id                         int64
patient_id                       object
age                               int64
gender                           object
blood_group                      object
department                       object
diagnosis                        object
appointment_date                 object
waiting_days                      int64
previous_appointments             int64
missed_previous_appointments      int64
appointment_status               object
admitted                          int64
room_type                        object
length_of_stay_days               int64
previous_admissions               int64
systolic_bp                       int64
diastolic_bp                      int64
blood_sugar_mg_dl                 int64
cholesterol_mg_dl                 int64
bmi                             float64
lab_tests_count                   int64
treatments_count                  int64
consultation_fee_lkr              int64
room_charge_lkr    

## Section 2 — Missing Value Handling

**Diagnostic Finding:**  
Only `room_type` exhibits missing values ($906 / 1000$ records, $90.6\%$).

**Methodological Justification:**  
A single blanket imputation strategy (e.g., dropping rows or blindly filling with mode) is methodologically flawed here because the missingness has dual causes:
1. **Non-admitted Patients (`admitted == 0`):** Outpatients never occupied a hospital room. This missingness is **Missing Not At Random (MNAR)** by operational design. Imputing with an explicit category (`"Not Admitted"`) preserves structural domain truth.
2. **Admitted Patients (`admitted == 1`):** Admitted patients with an empty `room_type` represent genuine missing data (**Missing At Random - MAR**). We impute these rows using the **mode room type** among admitted patients (`"General Ward"`).

In [5]:
print("Missing values per column before handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Breakdown by admission status
print("\nMissing room_type broken down by admission status:")
print(df.groupby('admitted')['room_type'].apply(lambda x: x.isnull().sum()))

Missing values per column before handling:
room_type    906
dtype: int64

Missing room_type broken down by admission status:
admitted
0    670
1    236
Name: room_type, dtype: int64


In [6]:
# Step 2a: Non-admitted patients -> 'Not Admitted' (MNAR)
mask_not_admitted = (df['admitted'] == 0) & (df['room_type'].isnull())
df.loc[mask_not_admitted, 'room_type'] = 'Not Admitted'
print(f"Rows set to 'Not Admitted': {mask_not_admitted.sum()}")

# Step 2b: Admitted patients with missing room_type -> Mode Imputation (MAR)
admitted_mode = df.loc[(df['admitted'] == 1) & (df['room_type'].notnull()), 'room_type'].mode()[0]
mask_admitted_missing = (df['admitted'] == 1) & (df['room_type'].isnull())
df.loc[mask_admitted_missing, 'room_type'] = admitted_mode
print(f"Rows imputed with mode ('{admitted_mode}'): {mask_admitted_missing.sum()}")

# Verification Assertion
assert df['room_type'].isnull().sum() == 0, "Error: room_type still has null values!"
print(f"\nRemaining missing values in dataset: {df.isnull().sum().sum()} total.")
print("  All missing values successfully resolved.")

Rows set to 'Not Admitted': 670
Rows imputed with mode ('General Ward'): 236

Remaining missing values in dataset: 0 total.
  All missing values successfully resolved.


## Section 3 — Duplicate Record Detection

Even when data appears clean, duplicate verification must be explicitly audited across three analytical granularities:
1. **Full-row duplicates:** Redundant observations.
2. **`record_id` duplicates:** Unique primary key integrity.
3. **`patient_id` duplicates:** Multiple clinical visits per patient (informational only; not an error).

In [7]:
full_duplicates = df.duplicated().sum()
duplicate_record_ids = df['record_id'].duplicated().sum()
duplicate_patient_ids = df['patient_id'].duplicated().sum()

print(f"Full-row duplicates: {full_duplicates}")
print(f"Duplicate record_id values: {duplicate_record_ids}")
print(f"Repeated patient_id values (multiple visits): {duplicate_patient_ids}")

if full_duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dropped {full_duplicates} duplicate rows. New shape: {df.shape}")
else:
    print("  No duplicate records present. Dataset integrity maintained.")

Full-row duplicates: 0
Duplicate record_id values: 0
Repeated patient_id values (multiple visits): 0
  No duplicate records present. Dataset integrity maintained.


## Section 4 — Outlier Identification

**Approach:**  
We employ the Interquartile Range (IQR) method on continuous physiological variables:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

**Clinical Decision — Outliers are Retained, Not Removed or Capped:**  
In disease risk classification, extreme clinical measurements (e.g., severe hyperglycemia $> 200\text{ mg/dL}$, hypercholesterolemia $> 300\text{ mg/dL}$, severe obesity $\text{BMI} > 35$) represent critical pathological signals directly driving the **High Risk** diagnosis. Truncating or deleting these values would discard vital discriminatory information.

In [8]:
def iqr_outlier_report(series, col_name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"{col_name:22s} | bounds=({lower:.1f}, {upper:.1f}) | outliers={len(outliers)}")
    return outliers.index

clinical_cols = ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi']

outlier_indices = {}
print("=== Clinical Biomarker Outlier Audit ===")
for col in clinical_cols:
    outlier_indices[col] = iqr_outlier_report(df[col], col)

=== Clinical Biomarker Outlier Audit ===
age                    | bounds=(-3.0, 93.0) | outliers=0
systolic_bp            | bounds=(84.0, 172.0) | outliers=3
diastolic_bp           | bounds=(51.0, 107.0) | outliers=3
blood_sugar_mg_dl      | bounds=(44.5, 184.5) | outliers=10
cholesterol_mg_dl      | bounds=(105.0, 305.0) | outliers=6
bmi                    | bounds=(14.3, 36.7) | outliers=9


In [9]:
# Cross-check: Confirm outliers correlate predominantly with the 'High' risk label
print("\nRisk Level Distribution for Outlier Rows (Justification to Retain):")
for col in ['bmi', 'cholesterol_mg_dl', 'blood_sugar_mg_dl']:
    idx = outlier_indices[col]
    if len(idx) > 0:
        print(f"\n{col} outliers breakdown:")
        print(df.loc[idx, 'disease_risk_level'].value_counts())


Risk Level Distribution for Outlier Rows (Justification to Retain):

bmi outliers breakdown:
disease_risk_level
High      5
Medium    3
Low       1
Name: count, dtype: int64

cholesterol_mg_dl outliers breakdown:
disease_risk_level
High      3
Medium    2
Low       1
Name: count, dtype: int64

blood_sugar_mg_dl outliers breakdown:
disease_risk_level
High      9
Medium    1
Name: count, dtype: int64


In [ ]:
# Operational sanity check: Verify room charge daily rate legitimacy
paid_rows = df[df['room_charge_lkr'] > 0].copy()
paid_rows['rate_per_day'] = paid_rows['room_charge_lkr'] / paid_rows['length_of_stay_days']
print("Room Charge Rate per Day by room_type:")
print(paid_rows.groupby('room_type')['rate_per_day'].describe()[['mean', 'std', 'min', 'max']])
print(" Std = 0 confirms room charges follow strict daily billing tariffs (no data corruption).")

## Section 5 — Data Cleaning

Three domain inconsistencies are audited and rectified systematically without dropping rows:

| Clinical / Logical Inconsistency | Rows Affected | Remediation Action | Technical / Clinical Justification |
| :--- | :---: | :--- | :--- |
| $\text{missed\_appointments} > \text{previous\_appointments}$ | 16 | Cap at `previous_appointments` | Mathematically impossible to miss more appointments than scheduled. |
| $\text{systolic\_bp} \le \text{diastolic\_bp}$ | 3 | Swap values in place | Physiologically invalid; treated as a data-entry transposition error. |
| $\text{treatments} = 0 \text{ but } \text{medicine\_charge} > 0$ | 212 | Documented only (No action) | Billing discrepancy unrelated to disease risk predictors (out of scope for Option C). |

In [10]:
# Correction 1: Cap missed appointments
mask_bad_missed = df['missed_previous_appointments'] > df['previous_appointments']
print(f"Rows with impossible missed-appointment count: {mask_bad_missed.sum()}")

df.loc[mask_bad_missed, 'missed_previous_appointments'] = df.loc[mask_bad_missed, 'previous_appointments']
assert (df['missed_previous_appointments'] <= df['previous_appointments']).all()
print("  Fixed: missed_previous_appointments capped at previous_appointments.")

Rows with impossible missed-appointment count: 16
  Fixed: missed_previous_appointments capped at previous_appointments.


In [11]:
# Correction 2: Swap transposed Blood Pressure readings
mask_bad_bp = df['systolic_bp'] <= df['diastolic_bp']
print(f"Rows with invalid BP (systolic <= diastolic): {mask_bad_bp.sum()}")
print(df.loc[mask_bad_bp, ['record_id', 'systolic_bp', 'diastolic_bp']])

# Swap values
df.loc[mask_bad_bp, ['systolic_bp', 'diastolic_bp']] = \
    df.loc[mask_bad_bp, ['diastolic_bp', 'systolic_bp']].values

assert (df['systolic_bp'] > df['diastolic_bp']).all()
print(" Fixed: Systolic/Diastolic readings swapped. 0 rows dropped.")

Rows with invalid BP (systolic <= diastolic): 3
     record_id  systolic_bp  diastolic_bp
99         100           85            86
331        332           98           103
374        375           94            97
 Fixed: Systolic/Diastolic readings swapped. 0 rows dropped.


In [12]:
# Audit 3: Document billing inconsistency
mask_billing_inconsistency = (df['treatments_count'] == 0) & (df['medicine_charge_lkr'] > 0)
print(f"Billing inconsistency rows (documented only): {mask_billing_inconsistency.sum()}")
print("Rationale: Billing fields are excluded from Option C modeling, hence no distortion occurs.")

print(f"\n--- Data Cleaning Audit Summary ---")
print(f"Final active rows: {len(df)} (Original: {len(df_raw)} | Rows dropped: 0)")

Billing inconsistency rows (documented only): 212
Rationale: Billing fields are excluded from Option C modeling, hence no distortion occurs.

--- Data Cleaning Audit Summary ---
Final active rows: 1000 (Original: 1000 | Rows dropped: 0)


## Section 6 — Feature Encoding

* **Nominal Categorical Predictors** (`gender`, `blood_group`, `department`, `diagnosis`, `room_type`, `payment_method`, `payment_status`, `appointment_status`):  
  Transformed using **One-Hot Encoding**. Because these categories possess no inherent order, label encoding would impose arbitrary numeric distances that distort linear and distance-based algorithms.
* **Target Variable (`disease_risk_level`):**  
  Transformed using **Ordinal Encoding** (`Low: 0`, `Medium: 1`, `High: 2`) because clinical risk possesses a genuine hierarchical progression.

In [13]:
# Preserve readable labels for evaluation & reporting
df['disease_risk_level_label'] = df['disease_risk_level']

# Ordinal target mapping
risk_order = {'Low': 0, 'Medium': 1, 'High': 2}
df['disease_risk_level'] = df['disease_risk_level_label'].map(risk_order)

print("Target Encoding Verification:")
print(df[['disease_risk_level_label', 'disease_risk_level']].drop_duplicates())

Target Encoding Verification:
   disease_risk_level_label  disease_risk_level
0                      High                   2
1                    Medium                   1
12                      Low                   0


In [14]:
# One-hot encode nominal predictor columns
nominal_cols = [
    'gender', 'blood_group', 'department', 'diagnosis', 'room_type',
    'payment_method', 'payment_status', 'appointment_status'
]

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

print(f"Shape before encoding: {df.shape}")
print(f"Shape after encoding:  {df_encoded.shape}")
print(f"New one-hot features generated: {df_encoded.shape[1] - df.shape[1] + len(nominal_cols)}")

Shape before encoding: (1000, 34)
Shape after encoding:  (1000, 68)
New one-hot features generated: 42


## Section 7 — Feature Scaling Strategy

`StandardScaler` ($z = \frac{x - \mu}{\sigma}$) is initialized for all continuous numeric predictors. Standardizing feature scales is essential for Logistic Regression, SVM, and KNN to prevent high-magnitude features (e.g., `cholesterol_mg_dl` $\sim 150\text{--}350$) from dominating lower-magnitude features (e.g., `bmi` $\sim 18\text{--}38$).

**Critical Anti-Leakage Protocol:**  
The scaler is **fit strictly on the training partition only** after the train/test split in Section 10. Fitting across the entire dataset before splitting leaks test set variance into the training pipeline.

In [15]:
numeric_cols_to_scale = [
    'age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
    'cholesterol_mg_dl', 'bmi', 'previous_admissions'
]

scaler = StandardScaler()
print("StandardScaler initialized for:", numeric_cols_to_scale)
print(" Protocol: scaler.fit() deferred to Section 10 on X_train only to prevent Data Leakage.")

StandardScaler initialized for: ['age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'previous_admissions']
 Protocol: scaler.fit() deferred to Section 10 on X_train only to prevent Data Leakage.


## Section 8 — Feature Selection

Feature selection is driven by correlation analysis and clinical relevance to eliminate noise, reduce dimensionality, and prevent target leakage:

| Feature Group | Excluded Columns | Justification |
| :--- | :--- | :--- |
| **Identifiers** | `record_id`, `patient_id` | Non-predictive unique keys; risk of severe memorization. |
| **Orthogonal AI Targets** | `no_show`, `readmitted_30_days` | Targets for Options A & B; cross-target leakage risk. |
| **Financial / Administrative** | `total_bill_lkr`, `consultation_fee_lkr`, `room_charge_lkr`, `lab_charge_lkr`, `medicine_charge_lkr`, `waiting_days` | Near-zero correlation ($r < 0.05$); financial data generated post-diagnosis. |
| **Retained Predictors** | `age`, `systolic_bp`, `diastolic_bp`, `blood_sugar_mg_dl`, `cholesterol_mg_dl`, `bmi`, `previous_admissions`, Encoded Departments/Diagnoses | Strong physiological and clinical correlation ($r = 0.17 \text{ to } 0.54$). |

In [16]:
# Pearson correlation with target
numeric_for_corr = [
    'age', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl',
    'cholesterol_mg_dl', 'bmi', 'waiting_days', 'previous_appointments',
    'missed_previous_appointments', 'previous_admissions',
    'lab_tests_count', 'treatments_count', 'total_bill_lkr',
    'disease_risk_level'
]

corr_with_target = df[numeric_for_corr].corr()['disease_risk_level'].sort_values(ascending=False)
print("Correlation of Numeric Features with disease_risk_level:")
print(corr_with_target)

Correlation of Numeric Features with disease_risk_level:
disease_risk_level              1.000000
age                             0.537582
blood_sugar_mg_dl               0.474440
cholesterol_mg_dl               0.446766
bmi                             0.359904
systolic_bp                     0.310182
previous_admissions             0.171077
diastolic_bp                    0.085260
previous_appointments           0.049035
missed_previous_appointments    0.027006
waiting_days                    0.018815
total_bill_lkr                  0.011739
lab_tests_count                -0.001361
treatments_count               -0.002355
Name: disease_risk_level, dtype: float64


In [17]:
# Drop non-predictive, administrative, and leakage columns
drop_cols = [
    'record_id', 'patient_id',
    'no_show', 'readmitted_30_days',
    'total_bill_lkr', 'lab_tests_count', 'treatments_count', 'waiting_days',
    'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr',
    'appointment_date',
    'disease_risk_level_label'
]

df_selected = df_encoded.drop(columns=drop_cols)
print(f"Shape before selection: {df_encoded.shape}")
print(f"Shape after selection:  {df_selected.shape}")
print(f"\nRemaining feature count: {df_selected.shape[1]}")

Shape before selection: (1000, 68)
Shape after selection:  (1000, 54)

Remaining feature count: 54


## Section 9 — Feature Engineering

Continuous physiological measurements often interact non-linearly with disease severity. We construct clinical biomarker bands and composite scores based on established medical literature (AHA & WHO guidelines):

| Engineered Feature | Derivation / Clinical Formula | Clinical Utility |
| :--- | :--- | :--- |
| **`bp_category`** | Normal ($<120 / <80$), Elevated ($120\text{--}129 / <80$), Hypertensive ($\ge 130 \text{ or } \ge 80$) | Captures cardiovascular hypertension thresholds (AHA guidelines). |
| **`bmi_category`** | Underweight ($<18.5$), Normal ($18.5\text{--}24.9$), Overweight ($25\text{--}29.9$), Obese ($\ge 30$) | Stratifies metabolic adiposity risk (WHO bands). |
| **`blood_sugar_category`** | Normal ($<100$), Prediabetic ($100\text{--}125$), Diabetic ($\ge 126\text{ mg/dL}$) | Identifies impaired glucose tolerance and diabetic risk. |
| **`age_group`** | Child ($<13$), Young Adult ($13\text{--}29$), Adult ($30\text{--}59$), Senior ($\ge 60$) | Captures age-related physiological degeneration. |
| **`health_burden_score`** | $\text{previous\_admissions} + \text{previous\_appointments}$ | Quantifies cumulative historical healthcare utilization. |

In [18]:
def bp_category(row):
    s, d = row['systolic_bp'], row['diastolic_bp']
    if s < 120 and d < 80: return 'Normal'
    elif s < 130 and d < 80: return 'Elevated'
    else: return 'Hypertensive'

def bmi_category(bmi):
    if bmi < 18.5: return 'Underweight'
    elif bmi < 25: return 'Normal'
    elif bmi < 30: return 'Overweight'
    else: return 'Obese'

def blood_sugar_category(bs):
    if bs < 100: return 'Normal'
    elif bs < 126: return 'Prediabetic'
    else: return 'Diabetic'

def age_group(age):
    if age < 13: return 'Child'
    elif age < 30: return 'Young Adult'
    elif age < 60: return 'Adult'
    else: return 'Senior'

df['bp_category'] = df.apply(bp_category, axis=1)
df['bmi_category'] = df['bmi'].apply(bmi_category)
df['blood_sugar_category'] = df['blood_sugar_mg_dl'].apply(blood_sugar_category)
df['age_group'] = df['age'].apply(age_group)
df['health_burden_score'] = df['previous_admissions'] + df['previous_appointments']

print("Engineered Feature Previews:")
df[['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group', 'health_burden_score']].head()

Engineered Feature Previews:


,bp_category,bmi_category,blood_sugar_category,age_group,health_burden_score
0,Elevated,Overweight,Prediabetic,Adult,2
1,Hypertensive,Obese,Diabetic,Young Adult,3
2,Hypertensive,Overweight,Normal,Young Adult,8
3,Hypertensive,Normal,Diabetic,Adult,1
4,Hypertensive,Overweight,Normal,Adult,5


In [19]:
# One-hot encode engineered categorical features and merge
df_final = df_selected.copy()
engineered_categoricals = ['bp_category', 'bmi_category', 'blood_sugar_category', 'age_group']

for col in engineered_categoricals:
    dummies = pd.get_dummies(df[col], prefix=col)
    df_final = pd.concat([df_final, dummies], axis=1)

df_final['health_burden_score'] = df['health_burden_score']

print(f"Final shape after feature engineering: {df_final.shape}")
df_final.head()

Final shape after feature engineering: (1000, 69)


,age,previous_appointments,missed_previous_appointments,admitted,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,disease_risk_level,gender_Female,gender_Male,blood_group_A+,blood_group_A-,blood_group_AB+,blood_group_AB-,blood_group_B+,blood_group_B-,blood_group_O+,blood_group_O-,department_Cardiology,department_General Medicine,department_Laboratory Services,department_Neurology,department_Orthopedics,department_Pediatrics,department_Radiology,diagnosis_Asthma,diagnosis_Back Pain,diagnosis_Chest Pain,diagnosis_Diabetes,diagnosis_Fever,diagnosis_Fracture,diagnosis_Hypertension,diagnosis_Kidney Infection,diagnosis_Migraine,diagnosis_Pneumonia,room_type_General Ward,room_type_ICU,room_type_Not Admitted,room_type_Private Room,payment_method_Card,payment_method_Cash,payment_method_Insurance,payment_method_Online,payment_status_Paid,payment_status_Partially Paid,payment_status_Unpaid,appointment_status_Cancelled,appointment_status_Completed,appointment_status_No-Show,appointment_status_Scheduled,bp_category_Elevated,bp_category_Hypertensive,bp_category_Normal,bmi_category_Normal,bmi_category_Obese,bmi_category_Overweight,bmi_category_Underweight,blood_sugar_category_Diabetic,blood_sugar_category_Normal,blood_sugar_category_Prediabetic,age_group_Adult,age_group_Child,age_group_Senior,age_group_Young Adult,health_burden_score
0,53,1,0,0,0,1,127,75,117,211,26.1,2,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,True,False,False,False,2
1,26,3,1,0,0,0,130,73,136,173,32.8,1,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,3
2,22,7,1,0,0,1,141,64,90,176,29.4,1,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,False,True,8
3,44,1,0,0,0,0,124,82,126,189,24.9,1,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,1
4,51,4,0,0,0,1,119,81,65,195,27.0,1,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,True,False,False,False,5


## Section 10 — Class Imbalance Check, Train/Test Split, and Scaling

* **Class Distribution:** Medium ($46.9\%$), High ($40.0\%$), Low ($13.1\%$).
* **Stratified Splitting ($80/20$):** We enforce `stratify=y` to ensure the minority class (`Low Risk`) is represented identically in both training ($800$ samples) and testing ($200$ samples) sets.
* **Leakage-Free Scaling:** The scaler is fit strictly on `X_train`, and `scaler.transform()` is applied to both `X_train` and `X_test`.

In [20]:
X = df_final.drop(columns=['disease_risk_level'])
y = df_final['disease_risk_level']

print("Class Distribution across Full Dataset:")
print(y.value_counts(normalize=True).round(3) * 100)

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTrain Shape: {X_train.shape} | Test Shape: {X_test.shape}")
print("\nTrain Class Balance (%):")
print(y_train.value_counts(normalize=True).round(3) * 100)
print("\nTest Class Balance (%):")
print(y_test.value_counts(normalize=True).round(3) * 100)

Class Distribution across Full Dataset:
disease_risk_level
1    46.9
2    40.0
0    13.1
Name: proportion, dtype: float64

Train Shape: (800, 68) | Test Shape: (200, 68)

Train Class Balance (%):
disease_risk_level
1    46.9
2    40.0
0    13.1
Name: proportion, dtype: float64

Test Class Balance (%):
disease_risk_level
1    47.0
2    40.0
0    13.0
Name: proportion, dtype: float64


In [21]:
# Scaling continuous predictors (Train Fit Only)
scaler = StandardScaler()
X_train[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])

print(" Scaling applied successfully (fit on X_train only).")
X_train[numeric_cols_to_scale].describe().round(2)

 Scaling applied successfully (fit on X_train only).


,age,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,previous_admissions
count,800.00,800.00,800.00,800.00,800.00,800.00,800.00
mean,0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.45,-2.83,-2.89,-1.88,-2.84,-2.73,-0.90
25%,-0.68,-0.69,-0.70,-0.69,-0.70,-0.65,-0.90
50%,-0.01,-0.04,-0.01,-0.04,-0.04,-0.01,0.17
75%,0.65,0.68,0.69,0.66,0.69,0.64,0.17
max,2.48,3.22,3.17,3.34,3.40,3.03,4.42


## Section 11 — Artifact Serialization & Preprocessing Summary

We serialize the clean, scaled, and encoded datasets to `data/processed/` for direct consumption by:
* **Notebook 03:** Model Development (Zumra Hassan)
* **Notebook 04:** Model Evaluation (Nilupul Thisaranga)
* **Notebook 05:** Explainable AI & Prototype (Siluna Nusal)

In [22]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Export split matrices
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

# Export full preprocessed dataframe for reference
df_final.to_csv(PROCESSED_DIR / '/smartcare_clean_dataset.csv', index=False)

print("Files successfully saved to:", PROCESSED_DIR.resolve())
print("/content/smartcare_clean_dataset.csv")
print(" /content/X_train.csv")
print("/content/y_train.csv")

print(f"\n=== PREPROCESSING WORKFLOW SUMMARY ===")
print(f"• Original Records Ingested : {len(df_raw)}")
print(f"• Final Clean Records Retained: {len(df_final)} (0 rows dropped)")
print(f"• Total Features Created    : {X.shape[1]}")
print(f"• Training Samples (80%)    : {X_train.shape[0]}")
print(f"• Testing Samples (20%)     : {X_test.shape[0]}")

Files successfully saved to: /data/processed
/content/smartcare_clean_dataset.csv
 /content/X_train.csv
/content/y_train.csv

=== PREPROCESSING WORKFLOW SUMMARY ===
• Original Records Ingested : 1000
• Final Clean Records Retained: 1000 (0 rows dropped)
• Total Features Created    : 68
• Training Samples (80%)    : 800
• Testing Samples (20%)     : 200


## Data Cleaning & Feature Engineering Workflow Summary Table

| Pipeline Step | Specific Action Executed | Rows/Cols Affected | Rows Dropped |
| :--- | :--- | :--- | :---: |
| **Missing Values** | Split imputation: `"Not Admitted"` for OPD vs Mode (`"General Ward"`) for Inpatients | 906 rows | 0 |
| **Duplicates** | Checked 3 levels (full row, `record_id`, `patient_id`) — 0 duplicates found | — | 0 |
| **Outliers** | Detected via IQR; retained because extreme vitals represent high-risk clinical pathology | 6 clinical vitals | 0 |
| **Data Cleaning** | Capped 16 missed appointments, swapped 3 transposed BP records, documented 212 billing inconsistencies | 231 rows touched | 0 |
| **Encoding** | One-hot encoded 8 nominal predictors; ordinal encoded target (`Low: 0`, `Medium: 1`, `High: 2`) | 8 nominal cols | — |
| **Scaling** | `StandardScaler` fitted on `X_train` only, transforming `X_train` and `X_test` | 7 numeric cols | — |
| **Selection** | Dropped 13 columns (IDs, orthogonal targets, near-zero correlation billing variables) | 13 columns dropped | — |
| **Engineering** | Derived 5 clinical features (4 categorical bands + 1 cumulative health burden score) | +5 features | — |
| **Split** | Stratified 80/20 split preserving class proportions across train/test splits | 1000 rows | — |

**Outcome:** 1000 valid records retained, feature space engineered and standardized without target leakage — ready for Task 05 Model Training.